# API обслуживания /mcp/kb-admin

Endpoint для индексатора и сильных моделей: всё видно с метаданными, запись статьи целиком, защита очереди индексации через version. Здесь же — где лежат правила индексатора.

## Отличия от API потребителя

Для индексатора и сильных моделей-краулеров. То же, что API потребителя (профиль `full`), но **со всеми метаданными** и с записью статьи целиком.

- Видно всё: секции без заголовка, статьи без шапки, все поля `metadata.kb` (`indexed`, `hash`, `updated`, `changed_by`, теги с `source`).
- **Запись через админку ставит `indexed = true`** и `changed_by = indexer`; `hash` и `updated` сервер считает сам (клиенту не доверяем).
- Ограничения структуры (имена, глубина, шапки, уникальность `id` и заголовков в статье) проверяются так же.

### Формат ответов
Те же блоки, что у потребителя (плоский текст в `content`, JSON в `structuredContent`), плюс метаданные строками перед `---`:
```
version: 3f2a9c…

=== coding/python/asyncio#event-loop
title: Event loop
indexed: false
updated: 2026-11-20T10:15:00Z
changed_by: mcp
hash: 8b1d…
tags: event loop [indexer], concurrency [mcp]
---
Текст секции как есть...
```
- `version` — первой строкой ответа `get_full`; модель копирует её в `put_*` как есть.
- Секция без заголовка — `title:` с пустым значением.
- `unindexed` — строка на статью: `coding/python/asyncio — 3 unindexed — oldest 2026-11-01T08:00:00Z`.
- `conflict` — обычная ошибка: `ERROR conflict: …`, `Hint: call get_full("…") again`.

## Команды админки

| Команда | Что делает |
|---|---|
| `unindexed(path?)` | статьи, где есть секции с `indexed = false` (или статья/тема без шапки): путь, число неиндексированных секций, самый старый `updated`. Сортировка — от старых к новым |
| `get_full(path)` | статья целиком: все секции в порядке файла со всеми метаданными + `version` статьи |
| `put_article(path, sections, version)` | **перезаписать статью целиком одним запросом**: шапка + секции в нужном порядке, у каждой `id`, `title`, `content`, `tags` (строки — источник `indexer`, или `{text, source}`). Можно менять порядок, делить, склеивать, переименовывать `id`. Все секции получают `indexed = true` |
| `put_sections(path, sections, version)` | записать несколько секций с метаданными (по `id`; нет — добавить в конец); остальные не трогаются. Записанные получают `indexed = true` |
| команды потребителя (`full`) | доступны как есть |

- В `put_article` / `put_sections` у существующей секции `title` и `content` можно не передавать — остаются текущие (`{id, tags}` — правка только тегов). Для новой секции они обязательны, иначе `invalid_input`.

Типичный проход индексатора: `unindexed()` → для каждой статьи `get_full` → обработать статью целиком (порядок секций, деление/объединение, `id`, теги `indexer`, описание в шапке) → `put_article` с полученным `version`.

## Защита через version

Индексатор читает статью, долго думает (LLM), потом пишет. Если за это время статью поправили, запись поверх потеряет правку и пометит её «проиндексированной» — она выпадет из очереди.

- `get_full` отдаёт `version` — хеш **значимого содержимого** ноутбука (не байтов файла); `put_article` / `put_sections` передают его обратно как есть.
- Не совпало — `ERROR conflict`, статья не перезаписывается; `Hint: call get_full("…") again`. Индексатор берёт статью на следующем проходе.
- Это только защита очереди индексации, а не общий механизм конфликтов.

### Что входит в хеш
Хешируется копия ноутбука, в которой нерелевантные поля заменены плейсхолдерами — иначе простое открытие и сохранение статьи в JupyterLab (он дописывает `kernelspec` / `language_info`) давало ложный `conflict`.

| Учитывается | Заменяется плейсхолдером |
|---|---|
| порядок и тип ячеек, текст (`source`), вложения (`attachments`) | метаданные ноутбука (`kernelspec`, `language_info` и т.п.), `nbformat` / `nbformat_minor` |
| `metadata.kb.id`, `metadata.kb.tags` и прочие поля `kb` | родной `id` ячейки Jupyter, метаданные ячейки кроме `kb` (свёртка и прочее состояние интерфейса) |
| | в `metadata.kb`: `updated`, `changed_by`, `indexed`, `hash` — производные от текста или служебные |

Изменение текста, порядка секций, `id` или тегов (в т.ч. теги потребителя) меняет `version`; остальное — нет. Реализация — `content_version` в `kb_mcp/kb.py`.

## Правила индексатора

Правила работы индексатора — не в этой статье, а в текстах самого endpoint-а: `jupyter_data/mcp_docs/<lang>/admin.ipynb` (устройство файлов и загрузка — `development/mcp-kb/plugin#mcp-docs`). Прочитать без подключения: `GET /mcp/docs/ru/admin`.

Ключевые правила (кратко):
- проход: `unindexed()` → `get_full` → обработка статьи целиком → `put_article` с тем же `version`; `conflict` — пропустить статью до следующего прохода;
- `put_article` пишет статью **целиком**: пропущенная секция удаляется;
- переданный `tags` **заменяет все** теги секции: теги потребителя (`mcp`) вернуть с `source`, или не передавать `tags`, чтобы оставить как есть;
- менять только теги — передавать `{id, tags}` без `title` / `content`;
- менять форму, навигацию и теги, но не смысл и факты; удаление и перенос — только в отчёте.

Для Claude Code — MCP-сервер `kb-admin` и навык `kb-index` (порядок работы: очередь, затем уборка тегов, шапок, ссылок и структуры): его текст — статья `development/mcp-kb/skills/kb-index`, агент ставит его сам по её `#install`.